# Assignment 02: Image Noise and Denoising Techniques

## Task 1.1: Describe Image Noises

This section covers the fundamental types of image noise commonly encountered in digital image processing.

### Gaussian Noise

**Gaussian Noise** is one of the most common types of noise in images, characterized by random pixel intensity variations that follow a normal (Gaussian) distribution.

**Characteristics:**
- Follows a normal probability distribution
- Appears as random variations in pixel intensity across the entire image
- Can be positive or negative
- Spread is uniform across all frequency components
- Common in camera sensors and analog-to-digital conversion

**Mathematical Model:**
$$I_{noisy}(x, y) = I_{original}(x, y) + N(0, \sigma^2)$$

where $N(0, \sigma^2)$ represents random samples from a Gaussian distribution with mean 0 and variance $\sigma^2$.

### Salt and Pepper Noise

**Salt and Pepper Noise**, also known as impulse noise, consists of random pixels corrupted with either very high (white/salt) or very low (black/pepper) intensity values.

**Characteristics:**
- Random pixels are set to either maximum (white/255) or minimum (black/0) intensity
- Creates isolated black and white spots in the image
- Typically affects only 1-10% of pixels
- Results from errors in data transmission or compression
- Non-Gaussian in nature; represents extreme outliers

**Mathematical Model:**
$$I_{noisy}(x, y) = \begin{cases} 
0 & \text{with probability } p_1/2 \\
255 & \text{with probability } p_1/2 \\
I_{original}(x, y) & \text{with probability } 1-p_1
\end{cases}$$

where $p_1$ is the noise probability (typically 0.01-0.1).

### Speckle Noise

**Speckle Noise** is a multiplicative type of noise that degrades the image by multiplying the original signal with random values, commonly found in radar and ultrasound imaging.

**Characteristics:**
- Multiplicative noise (not additive like Gaussian)
- Appears as a grainy pattern that varies with image intensity
- Larger grain size in brighter regions, smaller in darker regions
- Results from interference in coherent imaging systems (radar, sonar, ultrasound)
- Difficult to remove without distorting the image

**Mathematical Model:**
$$I_{noisy}(x, y) = I_{original}(x, y) \times N(1, \sigma^2)$$

or in additive form:

$$I_{noisy}(x, y) = I_{original}(x, y) + I_{original}(x, y) \times N(0, \sigma^2)$$

where $N$ represents random noise samples from a specified distribution.

### Poisson Noise

**Poisson Noise**, also called shot noise, represents random variations in photon arrival at image sensors, where the noise follows a Poisson probability distribution.

**Characteristics:**
- Related to the quantum nature of light (photon counting)
- Intensity of noise is proportional to the signal intensity
- More prominent in low-light conditions (bright areas have more noise than dark areas)
- Inherent in the image acquisition process with CCDs and CMOS sensors
- Follows Poisson probability distribution: $P(X=k) = \frac{\lambda^k e^{-\lambda}}{k!}$

**Mathematical Model:**
$$I_{noisy}(x, y) \sim \text{Poisson}\left(\lambda = I_{original}(x, y)\right)$$

**Key Difference from Gaussian:**
- Gaussian noise has constant variance regardless of intensity
- Poisson noise has variance equal to intensity: $\text{Var} = \lambda = I(x, y)$
- In high-intensity regions, Poisson noise appears approximately Gaussian with $\sigma^2 = I(x, y)$

## Task 1.2: Apply Noise to an Image

In this task, we will load a sample grayscale image and apply each of the four noise types separately to demonstrate how each type of noise affects the image quality.

In [1]:
# Import necessary libraries
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Display settings
plt.rcParams['figure.figsize'] = (15, 12)
plt.rcParams['font.size'] = 10

In [ ]:
import cv2
import matplotlib.pyplot as plt
from google.colab import drive
import os

# Step 1: Mount Google Drive
drive.mount('/content/drive')

# Step 2: Correct path (IMPORTANT)
image_path = "/content/drive/MyDrive/gray.png"  
# OR if inside folders:
# image_path = "/content/drive/MyDrive/Deep-Learning-Projects/Noise Extraction/Images/gray.png"

# Step 3: Debug check
print("Path:", image_path)
print("Exists:", os.path.exists(image_path))

# Step 4: Load image
original_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

# Step 5: Verify
if original_image is None:
    print(f"❌ Error: Could not load image from {image_path}")
else:
    print("✅ Image loaded successfully!")
    print(f"Shape: {original_image.shape}")
    print(f"Dtype: {original_image.dtype}")
    print(f"Range: [{original_image.min()}, {original_image.max()}]")

    # Display
    plt.figure(figsize=(5, 5))
    plt.imshow(original_image, cmap='gray')
    plt.title('Original Grayscale Image')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Task 1.2.1: Apply Gaussian Noise
# Add Gaussian (white) noise with mean=0 and standard deviation=25

gaussian_noise = np.random.normal(loc=0, scale=25, size=original_image.shape)
gaussian_noisy_image = original_image.astype(np.float32) + gaussian_noise

# Clip values to valid range [0, 255]
gaussian_noisy_image = np.clip(gaussian_noisy_image, 0, 255).astype(np.uint8)

# Display Gaussian noise result
plt.figure(figsize=(5, 5))
plt.imshow(gaussian_noisy_image, cmap='gray')
plt.title('Gaussian Noise Applied (σ=25)')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Gaussian Noisy Image - Min: {gaussian_noisy_image.min()}, Max: {gaussian_noisy_image.max()}")

In [ ]:
# Task 1.2.2: Apply Salt and Pepper Noise
# Random pixels set to either 0 (pepper/black) or 255 (salt/white)

salt_pepper_noisy_image = original_image.copy().astype(np.float32)
noise_probability = 0.05  # 5% of pixels affected

# Generate random noise mask
noise_mask = np.random.rand(*original_image.shape)

# Apply salt (white spots)
salt_mask = noise_mask < noise_probability / 2
salt_pepper_noisy_image[salt_mask] = 255

# Apply pepper (black spots)
pepper_mask = (noise_mask >= noise_probability / 2) & (noise_mask < noise_probability)
salt_pepper_noisy_image[pepper_mask] = 0

salt_pepper_noisy_image = salt_pepper_noisy_image.astype(np.uint8)

# Display Salt and Pepper noise result
plt.figure(figsize=(5, 5))
plt.imshow(salt_pepper_noisy_image, cmap='gray')
plt.title(f'Salt and Pepper Noise Applied (p={noise_probability})')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Salt and Pepper Noisy Image - Min: {salt_pepper_noisy_image.min()}, Max: {salt_pepper_noisy_image.max()}")

In [ ]:
# Task 1.2.3: Apply Speckle Noise
# Multiplicative noise: I_noisy = I_original * N(1, sigma^2)

speckle_noisy_image = original_image.astype(np.float32)
speckle_noise = np.random.normal(loc=1, scale=0.1, size=original_image.shape)

# Multiply original image by noise (multiplicative)
speckle_noisy_image = speckle_noisy_image * speckle_noise

# Clip values to valid range [0, 255]
speckle_noisy_image = np.clip(speckle_noisy_image, 0, 255).astype(np.uint8)

# Display Speckle noise result
plt.figure(figsize=(5, 5))
plt.imshow(speckle_noisy_image, cmap='gray')
plt.title('Speckle Noise Applied (σ=0.1, multiplicative)')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Speckle Noisy Image - Min: {speckle_noisy_image.min()}, Max: {speckle_noisy_image.max()}")

In [ ]:
# Task 1.2.4: Apply Poisson Noise (Shot Noise)
# Poisson noise: variance is proportional to pixel intensity

# Normalize image to [0, 1] for Poisson distribution
normalized_image = original_image.astype(np.float32) / 255.0

# Generate Poisson noise (note: parameter is lambda, which is the intensity)
# We scale the image intensity as the lambda parameter
poisson_noisy_image = np.random.poisson(normalized_image * 255) / 255.0 * 255

# Clip and convert to uint8
poisson_noisy_image = np.clip(poisson_noisy_image, 0, 255).astype(np.uint8)

# Display Poisson noise result
plt.figure(figsize=(5, 5))
plt.imshow(poisson_noisy_image, cmap='gray')
plt.title('Poisson Noise Applied (Shot Noise)')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Poisson Noisy Image - Min: {poisson_noisy_image.min()}, Max: {poisson_noisy_image.max()}")

In [ ]:
# Display all noisy images together for comparison

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Original image
axes[0, 0].imshow(original_image, cmap='gray')
axes[0, 0].set_title('Original Image', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

# Gaussian noise
axes[0, 1].imshow(gaussian_noisy_image, cmap='gray')
axes[0, 1].set_title('Gaussian Noise (σ=25)', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

# Salt and Pepper noise
axes[0, 2].imshow(salt_pepper_noisy_image, cmap='gray')
axes[0, 2].set_title('Salt and Pepper Noise (p=0.05)', fontsize=12, fontweight='bold')
axes[0, 2].axis('off')

# Speckle noise
axes[1, 0].imshow(speckle_noisy_image, cmap='gray')
axes[1, 0].set_title('Speckle Noise (σ=0.1)', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

# Poisson noise
axes[1, 1].imshow(poisson_noisy_image, cmap='gray')
axes[1, 1].set_title('Poisson Noise (Shot Noise)', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

# Hide the extra subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("All noise types applied and displayed successfully!")

In [ ]:
# Save all noisy images

# Create output directory if it doesn't exist
output_dir = Path('noisy_images')
output_dir.mkdir(exist_ok=True)

# Save original image
cv2.imwrite(str(output_dir / 'original_image.png'), original_image)
print(f"Saved: {output_dir / 'original_image.png'}")

# Save Gaussian noisy image
cv2.imwrite(str(output_dir / 'gaussian_noise.png'), gaussian_noisy_image)
print(f"Saved: {output_dir / 'gaussian_noise.png'}")

# Save Salt and Pepper noisy image
cv2.imwrite(str(output_dir / 'salt_pepper_noise.png'), salt_pepper_noisy_image)
print(f"Saved: {output_dir / 'salt_pepper_noise.png'}")

# Save Speckle noisy image
cv2.imwrite(str(output_dir / 'speckle_noise.png'), speckle_noisy_image)
print(f"Saved: {output_dir / 'speckle_noise.png'}")

# Save Poisson noisy image
cv2.imwrite(str(output_dir / 'poisson_noise.png'), poisson_noisy_image)
print(f"Saved: {output_dir / 'poisson_noise.png'}")

print("\nAll images saved successfully in 'noisy_images' directory!")